<a href="https://colab.research.google.com/github/nishithy13/ds2002-fa26/blob/main/notebooks/01-foundations/2026-10-09%20%E2%80%94%20Public_API_to_Clean_Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    for attempt in range(tries):
      r = requests.get(BASE, params = params, timeout = 10)

      if r.status_code == 200:
        return pd.DataFrame(r.json()['daily'])

      if r.status_code not in RETRY_STATUSES:
        r.raise_for_status()

        time.sleep(2 ** attempt)

    raise RuntimeError("Weather request failed after all attempts")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

try:
  wx = get_weather('2026-09-05', '2026-10-03')
  wx = wx[wx['time'].isin(['2026-09-05', '2026-10-03'])]

except Exception:
   used_fallback = True
   wx = FALLBACK.copy()

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
from os import WEXITED
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

if used_fallback:
  WEXITED['source'] = 'fallback sample'
else:
  wx['source'] = 'open-meteo'

wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns =
 {'time': 'date',
  'temperature_2m_max': 'temp_max',
  'precipitation_sum': 'precip_mm'})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
difference = round(wx['precip_mm'].max() - wx['precip_mm'].min(), 2)

print("September 5 precipitation:", wx[wx['date'] == '2026-09-05']['precip_mm'].values[0], "mm")
print("October 3 precipitation:", wx[wx['date'] == '2026-10-03']['precip_mm'].values[0], "mm")

if wx['precip_mm'].max() > wx['precip_mm'].min():
    print("The wetter date had", difference, "mm more precipitation.")


September 5 precipitation: 2.5 mm
October 3 precipitation: 14.7 mm
The wetter date had 12.2 mm more precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
print("Endpoint: https://archive-api.open-meteo.com/v1/archive")
print("Coordinates: latitude 38.03, longitude -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Date pulled: 2026-10-11")
print("Data source:", wx['source'].values[0])

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Date range: 2026-09-05 to 2026-10-03
Date pulled: 2026-10-11
Data source: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The chart might show missing weather data or an incorrect relationship between rain and revenue because the weather DataFrame is empty. Someone may not catch this becaue the code woudl run without an error, appearing to look valid with data missing.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
wx = get_weather('2026-09-05', '2026-10-03')
wx = wx[wx['time'].isin(['2026-09-05', '2026-10-03'])]
wx['city'] = 'Charlottesville'

orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando = orlando[orlando['time'].isin(['2026-09-05', '2026-10-03'])]
orlando['city'] = 'Orlando'

both_cities = pd.concat([wx, orlando], ignore_index=True)
both_cities

,time,temperature_2m_max,precipitation_sum,city
0,2026-09-05,29.4,2.5,Charlottesville
1,2026-10-03,22.0,14.7,Charlottesville
2,2026-09-05,32.1,3.9,Orlando
3,2026-10-03,30.4,14.5,Orlando
